# Assignment 1 — Python + Pandas: Data Exploration & Cleaning

**Objective:** Learn Python basics and perform basic data exploration and cleaning using Pandas.

**Steps covered in this notebook**

| # | Step | Section |
|---|------|---------|
| 1 | Load a CSV dataset into a Pandas DataFrame | 1 |
| 2 | Explore data (head/tail, shape, columns, dtypes) | 2 |
| 3 | Handle missing values (identify, fill / drop) | 3 |
| 4 | Basic operations (filter rows, select columns) | 4 |
| 5 | Remove duplicates | 5 |
| 6 | Create a derived column (`total_amount = price * quantity`) | 6 |
| 7 | Save the cleaned dataset as a new CSV | 7 |
| 8 | Summary | 8 |

**Dataset:** `data/superstore_orders.csv` — a Superstore-style order dataset that has been
seeded with **missing values, duplicate rows and untidy text** so the cleaning steps are real.

> Want to use the actual Kaggle Superstore file instead? See the note in Section 1.

## 0. Setup — imports and paths

In [2]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Resolve paths relative to the project root, so the notebook works no matter
# which folder VS Code launched the kernel from.
NB_DIR = os.getcwd()
PROJECT_ROOT = NB_DIR if os.path.isdir(os.path.join(NB_DIR, "data")) else os.path.dirname(NB_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("project:", PROJECT_ROOT)

ModuleNotFoundError: No module named 'numpy'

## 1. Load the CSV into a DataFrame

`pd.read_csv()` reads a comma-separated file into a **DataFrame** — a 2-D labelled table,
roughly a spreadsheet with a Python API.

**To use the real Kaggle Superstore file instead:** download
`Sample - Superstore.csv` from
<https://www.kaggle.com/datasets/vivek468/superstore-dataset-final>, drop it into `data/`,
and change `CSV_PATH` below. The Kaggle file uses `Sales`/`Quantity` rather than
`unit_price`/`quantity`, so also update `PRICE_COL` and `QTY_COL`.

In [ ]:
CSV_PATH = os.path.join(DATA_DIR, "superstore_orders.csv")
PRICE_COL, QTY_COL = "unit_price", "quantity"

# encoding="latin-1" is needed for the original Kaggle file; harmless here.
df_raw = pd.read_csv(CSV_PATH, encoding="latin-1")

print(f"Loaded {CSV_PATH}")
print(f"Rows: {df_raw.shape[0]:,}   Columns: {df_raw.shape[1]}")
df_raw.head()

In [ ]:
# Work on a copy so `df_raw` stays available for before/after comparisons.
df = df_raw.copy()

## 2. Explore the data

Five things worth checking before touching anything: the **shape**, the **column names**,
the **dtypes**, a few **sample rows**, and the **summary statistics**.

In [ ]:
print("--- shape (rows, columns) ---")
print(df.shape)

print("\n--- column names ---")
print(list(df.columns))

print("\n--- data types ---")
print(df.dtypes)

In [ ]:
print("=== FIRST 5 ROWS (head) ===")
display(df.head())

print("=== LAST 5 ROWS (tail) ===")
display(df.tail())

print("=== RANDOM 5 ROWS (sample) ===")
display(df.sample(5, random_state=0))

In [ ]:
# .info() = dtypes + non-null counts + memory footprint, all in one place
df.info()

In [ ]:
# Numeric summary: count / mean / std / min / quartiles / max
display(df.describe().T)

# Categorical summary: count / unique / most-frequent value
display(df.describe(include="object").T)

In [ ]:
# How many distinct values does each column hold?
nunique = df.nunique().sort_values(ascending=False)
display(nunique.to_frame("unique_values"))

# Value counts for a couple of key categorical columns
for col in ["category", "region", "segment"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

## 3. Handle missing values

### 3.1 Identify

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(2),
})
missing = missing[missing["missing_count"] > 0].sort_values("missing_count", ascending=False)

print(f"Total missing cells: {int(df.isnull().sum().sum())}")
display(missing)

In [ ]:
# Optional visual: a bar chart of missing counts per column
import matplotlib.pyplot as plt

if not missing.empty:
    ax = missing["missing_count"].plot(kind="barh", figsize=(7, 3.5), color="#4C72B0")
    ax.set_title("Missing values per column (before cleaning)")
    ax.set_xlabel("count of nulls")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values to plot.")

### 3.2 Decide a strategy

There is no single right answer — the choice depends on the column:

| Column | Type | Strategy | Why |
|---|---|---|---|
| `unit_price` | numeric | fill with **median** | median resists the skew that a few high-priced items create |
| `quantity` | numeric | fill with **median**, cast to int | quantity must stay a whole number |
| `discount` | numeric | fill with **0** | a missing discount most plausibly means "no discount applied" |
| `city`, `segment` | categorical | fill with **"Unknown"** | keeps the row usable instead of throwing it away |
| `customer_name` | categorical | fill with **"Unknown"** | same |
| `order_id`, `product_id` | key | **drop the row** | a record with no key can't be joined or de-duplicated |

### 3.3 Apply

In [ ]:
before_rows = len(df)

# (a) Drop rows missing a critical key — these are unusable downstream.
key_cols = ["order_id", "product_id"]
df = df.dropna(subset=key_cols)
print(f"Dropped {before_rows - len(df)} row(s) missing {key_cols}")

# (b) Numeric columns -> median (or a sensible constant)
price_median = df[PRICE_COL].median()
qty_median = df[QTY_COL].median()

df[PRICE_COL] = df[PRICE_COL].fillna(price_median)
df[QTY_COL] = df[QTY_COL].fillna(qty_median)
df["discount"] = df["discount"].fillna(0)

print(f"unit_price nulls filled with median = {price_median:.2f}")
print(f"quantity   nulls filled with median = {qty_median:.0f}")

# (c) Categorical columns -> "Unknown"
for col in ["city", "segment", "customer_name"]:
    df[col] = df[col].fillna("Unknown")

# (d) Fix the dtype that the nulls had forced to float
df[QTY_COL] = df[QTY_COL].astype(int)

In [ ]:
# Tidy the text columns too: strip whitespace, normalise casing
text_cols = df.select_dtypes(include="object").columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())
df["ship_mode"] = df["ship_mode"].str.title()

print(df["ship_mode"].value_counts())

In [ ]:
# Parse the date columns (source uses day-first format: 31/12/2023)
for col in ["order_date", "ship_date"]:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

print(df[["order_date", "ship_date"]].dtypes)
print(f"\nDate range: {df['order_date'].min().date()} -> {df['order_date'].max().date()}")

In [ ]:
# Verify: no nulls should remain
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if remaining.empty:
    print("No missing values remain.")
else:
    display(remaining)

## 4. Basic operations — selecting columns and filtering rows

In [ ]:
# --- Selecting columns -------------------------------------------------
one_col = df["category"]                                  # -> Series
few_cols = df[["order_id", "category", PRICE_COL, QTY_COL]]  # -> DataFrame

print(type(one_col), "|", type(few_cols))
display(few_cols.head())

In [ ]:
# --- .loc (label-based) vs .iloc (position-based) ----------------------
print("First 3 rows, selected columns, via .loc:")
display(df.loc[df.index[:3], ["order_id", "city", PRICE_COL]])

print("First 3 rows, first 4 columns, via .iloc:")
display(df.iloc[:3, :4])

In [ ]:
# --- Filtering rows: a single condition --------------------------------
tech = df[df["category"] == "Technology"]
print(f"Technology orders: {len(tech)}")
display(tech.head(3))

In [ ]:
# --- Filtering rows: multiple conditions -------------------------------
# & = AND, | = OR, ~ = NOT. Each condition MUST be wrapped in parentheses.
high_value_west = df[(df[PRICE_COL] > 500) & (df["region"] == "West")]
print(f"Orders over $500 in the West region: {len(high_value_west)}")
display(high_value_west[["order_id", "region", "category", PRICE_COL, QTY_COL]].head())

# .isin() for membership, .between() for ranges, .str.contains() for substrings
subset = df[df["segment"].isin(["Corporate", "Home Office"]) & df[QTY_COL].between(5, 10)]
print(f"\nCorporate/Home-Office orders with quantity 5-10: {len(subset)}")

In [ ]:
# --- Sorting and a quick group-by --------------------------------------
top = df.sort_values(PRICE_COL, ascending=False).head(5)
display(top[["order_id", "product_name", PRICE_COL, QTY_COL]])

by_category = df.groupby("category").agg(
    orders=("order_id", "count"),
    avg_price=(PRICE_COL, "mean"),
    total_qty=(QTY_COL, "sum"),
).round(2)
display(by_category)

## 5. Remove duplicates

Two different notions of "duplicate" show up in this dataset:

1. **Exact duplicates** — every column identical. Always safe to drop.
2. **Business-key duplicates** — the same `order_id` + `product_id` appearing on more than
   one row. That combination should be unique, so these are drops too. Keeping the *last*
   occurrence is the usual choice when later rows are assumed to be more recent.

In [ ]:
exact_dupes = df.duplicated().sum()
key_dupes = df.duplicated(subset=["order_id", "product_id"]).sum()

print(f"Exact duplicate rows          : {exact_dupes}")
print(f"Duplicate (order_id, product_id): {key_dupes}")

if exact_dupes:
    print("\nExample of an exact duplicate:")
    display(df[df.duplicated(keep=False)].sort_values("order_id").head(4))

In [ ]:
rows_before = len(df)

# Step 1: drop exact duplicates
df = df.drop_duplicates()
after_exact = len(df)

# Step 2: drop business-key duplicates, keeping the last occurrence
df = df.drop_duplicates(subset=["order_id", "product_id"], keep="last")
after_key = len(df)

df = df.reset_index(drop=True)

print(f"Rows before            : {rows_before}")
print(f"After exact de-dup     : {after_exact}  (-{rows_before - after_exact})")
print(f"After key de-dup       : {after_key}  (-{after_exact - after_key})")
print(f"\nRemaining duplicates: {df.duplicated(subset=['order_id', 'product_id']).sum()}")

## 6. Create derived columns

The required one is `total_amount = price * quantity`. A few more are added because they
cost one line each and make the summary far more interesting.

In [ ]:
# THE REQUIRED DERIVED COLUMN
df["total_amount"] = (df[PRICE_COL] * df[QTY_COL]).round(2)

# A few extras
df["discount_amount"] = (df["total_amount"] * df["discount"]).round(2)
df["net_amount"] = (df["total_amount"] - df["discount_amount"]).round(2)
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)

# A categorical band built with pd.cut()
df["value_band"] = pd.cut(
    df["total_amount"],
    bins=[-0.01, 250, 1000, 3000, float("inf")],
    labels=["Low", "Medium", "High", "Premium"],
)

display(df[[PRICE_COL, QTY_COL, "total_amount", "discount",
            "discount_amount", "net_amount", "value_band"]].head(10))

In [ ]:
print(f"Total revenue (gross): {df['total_amount'].sum():,.2f}")
print(f"Total discount given: {df['discount_amount'].sum():,.2f}")
print(f"Total revenue (net)  : {df['net_amount'].sum():,.2f}")
print(f"Average order value  : {df['net_amount'].mean():,.2f}")

display(df["value_band"].value_counts().sort_index().to_frame("orders"))

In [ ]:
# Optional visual: net revenue by category
rev = df.groupby("category")["net_amount"].sum().sort_values()
ax = rev.plot(kind="barh", figsize=(7, 3), color="#55A868")
ax.set_title("Net revenue by category (cleaned data)")
ax.set_xlabel("net amount")
plt.tight_layout()
plt.show()

## 7. Save the cleaned dataset

In [ ]:
CLEAN_PATH = os.path.join(OUTPUT_DIR, "superstore_orders_cleaned.csv")
df.to_csv(CLEAN_PATH, index=False)

print(f"Saved -> {CLEAN_PATH}")
print(f"Size  : {os.path.getsize(CLEAN_PATH):,} bytes")

# Read it back to prove the round-trip works
check = pd.read_csv(CLEAN_PATH)
print(f"Re-read shape: {check.shape}")
check.head(3)

## 8. Summary

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Rows loaded (raw)",
        "Columns loaded (raw)",
        "Missing cells (raw)",
        "Exact duplicate rows removed",
        "Key duplicate rows removed",
        "Rows after cleaning",
        "Columns after cleaning (incl. derived)",
        "Missing cells remaining",
        "Gross revenue (total_amount)",
        "Net revenue (after discount)",
    ],
    "Value": [
        f"{df_raw.shape[0]:,}",
        f"{df_raw.shape[1]}",
        f"{int(df_raw.isnull().sum().sum()):,}",
        f"{rows_before - after_exact}",
        f"{after_exact - after_key}",
        f"{df.shape[0]:,}",
        f"{df.shape[1]}",
        f"{int(df.isnull().sum().sum())}",
        f"{df['total_amount'].sum():,.2f}",
        f"{df['net_amount'].sum():,.2f}",
    ],
})
display(summary)
print("\nAssignment 1 complete. Cleaned file is in output/superstore_orders_cleaned.csv")

### What was done, in words

1. **Loaded** `superstore_orders.csv` into a DataFrame with `pd.read_csv()`.
2. **Explored** it with `.shape`, `.columns`, `.dtypes`, `.head()`, `.tail()`, `.info()`,
   `.describe()` and `.value_counts()`.
3. **Handled missing values** column by column: median-fill for `unit_price` and `quantity`,
   zero-fill for `discount`, `"Unknown"` for the categorical columns, and row-drops for
   records missing a business key.
4. **Filtered and selected** with boolean masks, `.loc` / `.iloc`, `.isin()` and `.between()`.
5. **Removed duplicates** in two passes — exact rows first, then `(order_id, product_id)`.
6. **Derived** `total_amount = unit_price * quantity`, plus discount, net amount, shipping
   days and a `value_band` category.
7. **Saved** the result to `output/superstore_orders_cleaned.csv`.

Next: open `02_delta_lake_merge.ipynb` for the Delta Lake incremental / SCD assignment.